# 02_v2 Preprocessing Policy

Applied Stage 02 preprocessing for v2 Membership rows only. This notebook creates auditable interim Membership, UserMapping, and MovieMaster policy-checked outputs. It does not create usage features, content features, modeling datasets, SHAP outputs, or trained models.


In [1]:
import csv
import json
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "_data" / "01_raw"
STAGE01_TABLE_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "tables" / "01_v2_data_overview_and_audit"
STAGE01_DATA_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "01_v2_data_overview_and_audit"
VALID_TABLE_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "tables" / "02_v2_preprocessing_policy_validation"
VALID_DATA_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "02_v2_preprocessing_policy_validation"
INTERIM_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "02_v2_preprocessing_policy"
TABLE_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "tables" / "02_v2_preprocessing_policy"
DATA_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "02_v2_preprocessing_policy"

RAW_FILES = {
    "Membership": RAW_DIR / "Membership.csv",
    "User_Mapping": RAW_DIR / "User_Mapping.csv",
    "View_History": RAW_DIR / "View_History.csv",
    "Movie_Master": RAW_DIR / "Movie_Master.csv",
}
PROTECTED_DIRS = [STAGE01_TABLE_DIR, STAGE01_DATA_DIR, VALID_TABLE_DIR, VALID_DATA_DIR]
TARGET_COL = "is_repurchase"
CORE_EVENT_FIELDS = ["USER_KEY", "product_code", "price", "max_screen", "reg_date", "end_date", "payment_device", "billing_method"]
EXPECTED_SCREEN_VALUES = {1, 2, 3, 4}
FORBIDDEN_MODEL_FEATURES = ["USER_KEY", "USER_NUM", "MOVIE_NUM", "reg_date", "end_date", "duration_days", "watch_date", "is_repurchase"]

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

def snapshot_files(paths):
    snap = {}
    for path in paths:
        if path.is_file():
            files = [path]
        elif path.exists():
            files = [p for p in path.rglob("*") if p.is_file()]
        else:
            files = []
        for file in files:
            snap[str(file)] = {"size": file.stat().st_size, "mtime_ns": file.stat().st_mtime_ns}
    return snap

raw_before = snapshot_files(list(RAW_FILES.values()))
protected_before = snapshot_files(PROTECTED_DIRS)

def rel(path):
    return str(path.relative_to(PROJECT_ROOT)).replace("\\", "/")

def read_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        return reader.fieldnames or [], list(reader)

def write_csv(path, rows, fieldnames):
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({field: row.get(field, "") for field in fieldnames})

def write_json(path, payload):
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

def parse_date(value, fmt):
    return datetime.strptime(value, fmt).date()

def safe_int(value):
    try:
        return int(value)
    except Exception:
        return None

def safe_float(value):
    try:
        return float(value)
    except Exception:
        return None

def add_exclusion(exclusions, row, reason_code, reason_detail, source_rule_name):
    exclusions[row["membership_row_id"]] = {
        "membership_row_id": row["membership_row_id"],
        "source_row_number": row["source_row_number"],
        "USER_KEY": row.get("USER_KEY", ""),
        "is_repurchase": row.get(TARGET_COL, ""),
        "reason_code": reason_code,
        "reason_detail": reason_detail,
        "source_rule_name": source_rule_name,
    }

loaded = {}
for name, path in RAW_FILES.items():
    cols, rows = read_csv(path)
    loaded[name] = {"columns": cols, "rows": rows}

membership_cols = loaded["Membership"]["columns"]
membership_raw = []
for idx, row in enumerate(loaded["Membership"]["rows"], start=1):
    out = {"membership_row_id": idx, "source_file": "_data/01_raw/Membership.csv", "source_row_number": idx + 1}
    out.update(row)
    try:
        out["duration_days"] = (parse_date(row["end_date"], "%y-%m-%d") - parse_date(row["reg_date"], "%y-%m-%d")).days
        out["date_parse_status"] = "ok"
    except Exception as exc:
        out["duration_days"] = ""
        out["date_parse_status"] = f"parse_error:{exc}"
    membership_raw.append(out)
mapping = loaded["User_Mapping"]["rows"]
views = loaded["View_History"]["rows"]
movies = loaded["Movie_Master"]["rows"]

# 1-4. Applied Membership exclusions: strict target conflicts, exact duplicates, same non-target/same-target duplicates.
non_target_cols = [c for c in membership_cols if c != TARGET_COL]
strict_groups = defaultdict(list)
full_groups = defaultdict(list)
same_non_target_same_target_groups = defaultdict(list)
for row in membership_raw:
    strict_groups[tuple(row.get(c, "") for c in non_target_cols)].append(row)
    full_groups[tuple(row.get(c, "") for c in membership_cols)].append(row)
    same_non_target_same_target_groups[tuple(row.get(c, "") for c in non_target_cols + [TARGET_COL])].append(row)

strict_conflict_groups = {k: rows for k, rows in strict_groups.items() if len({r[TARGET_COL] for r in rows}) > 1}
excluded = {}
target_conflict_resolution_rows = []
for group_id, rows in enumerate(strict_conflict_groups.values(), start=1):
    ids = "|".join(str(r["membership_row_id"]) for r in rows)
    targets = "|".join(sorted({r[TARGET_COL] for r in rows}))
    target_conflict_resolution_rows.append({
        "conflict_group_id": group_id,
        "group_size": len(rows),
        "target_values": targets,
        "membership_row_ids": ids,
        "action": "exclude_all_rows_in_group",
        "reason_code": "STRICT_TARGET_CONFLICT",
        "reason_detail": "All non-target Membership fields are identical but is_repurchase differs.",
    })
    for row in rows:
        add_exclusion(excluded, row, "STRICT_TARGET_CONFLICT", "All non-target Membership fields are identical but is_repurchase differs.", "R_LABEL_01_EXCLUDE_STRICT_TARGET_CONFLICT")

duplicate_resolution_rows = []
removed_duplicate_rows = []
def resolve_duplicate_groups(groups, duplicate_type, reason_code, source_rule_name):
    group_index = len(duplicate_resolution_rows)
    removed_count = 0
    for rows in groups.values():
        active_rows = [r for r in rows if r["membership_row_id"] not in excluded]
        if len(active_rows) <= 1:
            continue
        active_rows = sorted(active_rows, key=lambda r: r["membership_row_id"])
        representative = active_rows[0]
        removed = active_rows[1:]
        group_index += 1
        duplicate_resolution_rows.append({
            "duplicate_group_id": group_index,
            "duplicate_type": duplicate_type,
            "group_size_after_prior_exclusions": len(active_rows),
            "kept_membership_row_id": representative["membership_row_id"],
            "removed_membership_row_ids": "|".join(str(r["membership_row_id"]) for r in removed),
            "removed_count": len(removed),
            "reason_code": reason_code,
            "source_rule_name": source_rule_name,
        })
        for row in removed:
            add_exclusion(excluded, row, reason_code, f"Duplicate resolved by keeping representative membership_row_id={representative['membership_row_id']}.", source_rule_name)
            rr = {"duplicate_type": duplicate_type, "kept_membership_row_id": representative["membership_row_id"], "reason_code": reason_code}
            rr.update(row)
            removed_duplicate_rows.append(rr)
        removed_count += len(removed)
    return removed_count

exact_duplicate_removed = resolve_duplicate_groups(full_groups, "exact_duplicate_all_membership_columns", "EXACT_DUPLICATE_EXTRA_ROW", "R_DUP_01_KEEP_FIRST_EXACT_DUPLICATE")
same_target_duplicate_removed = resolve_duplicate_groups(same_non_target_same_target_groups, "same_non_target_same_target_duplicate", "SAME_NON_TARGET_SAME_TARGET_DUPLICATE_EXTRA_ROW", "R_DUP_02_KEEP_FIRST_SAME_NON_TARGET_SAME_TARGET")

retained_membership = []
for row in membership_raw:
    if row["membership_row_id"] in excluded:
        continue
    retained = dict(row)
    retained["stage02_retained"] = "Y"
    retained["duration_policy_status"] = "DEFERRED"
    retained["not_model_feature_note"] = "Identifiers, raw dates, duration_days, and target are not model features in Stage 02."
    retained_membership.append(retained)

excluded_rows = [excluded[mid] for mid in sorted(excluded)]

# 6-7. Duration deferred and value anomaly flag audits.
duration_counts_raw = Counter(row["duration_days"] for row in membership_raw)
duration_counts_retained = Counter(row["duration_days"] for row in retained_membership)
duration_policy_deferred_summary = []
for value in sorted(set(duration_counts_raw) | set(duration_counts_retained), key=lambda x: str(x)):
    duration_policy_deferred_summary.append({
        "duration_days": value,
        "raw_membership_rows": duration_counts_raw.get(value, 0),
        "retained_membership_rows": duration_counts_retained.get(value, 0),
        "filter_applied": "N",
        "policy_status": "DEFERRED",
        "reason": "end_date inclusiveness and valid subscription-duration definition are not fully confirmed",
    })

value_anomaly_flag_summary = []
def add_value_rule(rule_id, field, reason_code, predicate, recommendation):
    raw_count = sum(1 for r in membership_raw if predicate(r))
    retained_count = sum(1 for r in retained_membership if predicate(r))
    value_anomaly_flag_summary.append({
        "rule_id": rule_id,
        "field": field,
        "reason_code": reason_code,
        "raw_affected_rows": raw_count,
        "retained_affected_rows": retained_count,
        "action": "audit_flag_only_no_delete",
        "recommendation": recommendation,
    })
add_value_rule("V_AGE_01", "age", "AGE_MISSING_OR_NON_NUMERIC", lambda r: safe_int(r.get("age", "")) is None, "flag; do not impute silently")
add_value_rule("V_AGE_02", "age", "AGE_OUTSIDE_0_100", lambda r: safe_int(r.get("age", "")) is not None and (safe_int(r.get("age", "")) < 0 or safe_int(r.get("age", "")) > 100), "ask mentor before correction")
add_value_rule("V_SCREEN_01", "max_screen", "MAX_SCREEN_MISSING_OR_NON_NUMERIC", lambda r: safe_int(r.get("max_screen", "")) is None, "flag; do not impute silently")
add_value_rule("V_SCREEN_02", "max_screen", "MAX_SCREEN_OUTSIDE_EXPECTED_1_4", lambda r: safe_int(r.get("max_screen", "")) is not None and safe_int(r.get("max_screen", "")) not in EXPECTED_SCREEN_VALUES, "ask mentor whether value is valid product")
add_value_rule("V_GENDER_01", "gender", "GENDER_MISSING", lambda r: r.get("gender", "") == "", "flag missing demographic")
add_value_rule("V_GENDER_02", "gender", "GENDER_UNEXPECTED_CATEGORY", lambda r: r.get("gender", "") not in {"", "M", "F"}, "ask mentor for codebook")
add_value_rule("V_VERIFY_01", "is_user_verified", "VERIFICATION_MISSING", lambda r: r.get("is_user_verified", "") == "", "flag missing verification")
add_value_rule("V_VERIFY_02", "is_user_verified", "VERIFICATION_UNEXPECTED_CATEGORY", lambda r: r.get("is_user_verified", "") not in {"", "Y", "N"}, "ask mentor for codebook")
add_value_rule("V_PROMO_01", "is_promotion", "PROMOTION_MISSING", lambda r: r.get("is_promotion", "") == "", "flag blank promotion until codebook confirms meaning")
add_value_rule("V_PROMO_02", "is_promotion", "PROMOTION_UNEXPECTED_CATEGORY", lambda r: r.get("is_promotion", "") not in {"", "O"}, "ask mentor for codebook")
add_value_rule("V_PROMO_03", "is_promotion", "PRICE_100_PROMOTION_MISMATCH", lambda r: r.get("price", "") == "100" and r.get("is_promotion", "") != "O", "ask mentor whether price=100 defines promotion")
add_value_rule("V_PRICE_01", "price", "PRICE_MISSING_OR_NON_NUMERIC", lambda r: safe_float(r.get("price", "")) is None, "ask mentor; price is core subscription field")
add_value_rule("V_PRICE_02", "price", "PRICE_NEGATIVE", lambda r: safe_float(r.get("price", "")) is not None and safe_float(r.get("price", "")) < 0, "ask mentor before exclusion")
add_value_rule("V_CHURNPREVENT_01", "is_churn_prevented", "CHURN_PREVENTED_MISSING", lambda r: r.get("is_churn_prevented", "") == "", "flag; do not use to correct target")
add_value_rule("V_CHURNPREVENT_02", "is_churn_prevented", "CHURN_PREVENTED_UNEXPECTED_CATEGORY", lambda r: r.get("is_churn_prevented", "") not in {"", "O"}, "ask mentor for codebook")
payment_device_counts = Counter(r.get("payment_device", "") for r in membership_raw)
billing_method_counts = Counter(r.get("billing_method", "") for r in membership_raw)
add_value_rule("V_PAYMENT_DEVICE_01", "payment_device", "PAYMENT_DEVICE_MISSING", lambda r: r.get("payment_device", "") == "", "flag missing payment device")
add_value_rule("V_PAYMENT_DEVICE_02", "payment_device", "PAYMENT_DEVICE_RARE", lambda r: payment_device_counts[r.get("payment_device", "")] <= 5, "rare category audit only")
add_value_rule("V_BILLING_METHOD_01", "billing_method", "BILLING_METHOD_MISSING", lambda r: r.get("billing_method", "") == "", "flag missing billing method")
add_value_rule("V_BILLING_METHOD_02", "billing_method", "BILLING_METHOD_RARE", lambda r: billing_method_counts[r.get("billing_method", "")] <= 5, "rare category audit only")

# 8. UserMapping policy checked, no Membership row multiplication applied.
key_to_nums = defaultdict(list)
num_to_keys = defaultdict(list)
for row in mapping:
    key_to_nums[row["USER_KEY"]].append(row["USER_NUM"])
    num_to_keys[row["USER_NUM"]].append(row["USER_KEY"])
retained_key_counts = Counter(row["USER_KEY"] for row in retained_membership)
usermapping_policy_checked = []
for idx, row in enumerate(mapping, start=1):
    user_key = row["USER_KEY"]
    user_num = row["USER_NUM"]
    distinct_nums = set(key_to_nums[user_key])
    distinct_keys = set(num_to_keys[user_num])
    out = {"source_row_number": idx + 1}
    out.update(row)
    out["USER_KEY_to_USER_NUM_pattern"] = "one_to_many" if len(distinct_nums) > 1 else "one_to_one"
    out["USER_NUM_to_USER_KEY_pattern"] = "many_to_one" if len(distinct_keys) > 1 else "one_to_one"
    out["retained_membership_event_count_for_USER_KEY"] = retained_key_counts[user_key]
    out["policy_note"] = "Do not multiply Membership rows; aggregate logs back to membership_row_id in Stage 03."
    usermapping_policy_checked.append(out)
usermapping_policy_summary = [
    {"metric": "raw_usermapping_rows", "count": len(mapping), "action": "kept_policy_checked", "note": "No rows removed."},
    {"metric": "USER_KEY_one_to_many_cases", "count": sum(1 for nums in key_to_nums.values() if len(set(nums)) > 1), "action": "audit_flag_only", "note": "Direct join can multiply rows."},
    {"metric": "USER_NUM_many_to_one_cases", "count": sum(1 for keys in num_to_keys.values() if len(set(keys)) > 1), "action": "audit_flag_only", "note": "If nonzero, ask mentor before USER_NUM grouping."},
    {"metric": "retained_membership_rows_with_multi_USER_NUM", "count": sum(1 for r in retained_membership if len(set(key_to_nums.get(r["USER_KEY"], []))) > 1), "action": "audit_flag_only", "note": "No usage aggregation created in Stage 02."},
]

# 9. MovieMaster policy checked, no content dedupe applied.
movie_cols = loaded["Movie_Master"]["columns"]
movie_by_num = defaultdict(list)
for idx, row in enumerate(movies, start=1):
    movie_by_num[row["MOVIE_NUM"]].append((idx + 1, row))
moviemaster_policy_checked = []
for idx, row in enumerate(movies, start=1):
    group = movie_by_num[row["MOVIE_NUM"]]
    conflict_cols = [c for c in movie_cols if len({r.get(c, "") for _, r in group}) > 1]
    out = {"source_row_number": idx + 1}
    out.update(row)
    out["duplicate_MOVIE_NUM_flag"] = "Y" if len(group) > 1 else "N"
    out["duplicate_group_size"] = len(group)
    out["conflicting_columns_within_MOVIE_NUM"] = "|".join(conflict_cols)
    out["policy_note"] = "Do not join Movie_Master to View_History in Stage 02; final content dedupe deferred to Stage 04."
    moviemaster_policy_checked.append(out)
moviemaster_policy_summary = [
    {"metric": "raw_moviemaster_rows", "count": len(movies), "action": "kept_policy_checked", "note": "No rows removed."},
    {"metric": "MOVIE_NUM_cardinality", "count": len(movie_by_num), "action": "audit", "note": "Movie id cardinality."},
    {"metric": "duplicate_MOVIE_NUM_count", "count": sum(1 for group in movie_by_num.values() if len(group) > 1), "action": "defer_final_deduplication", "note": "Stage 04 should deduplicate before content join."},
    {"metric": "duplicate_MOVIE_NUM_rows", "count": sum(len(group) for group in movie_by_num.values() if len(group) > 1), "action": "audit_flag_only", "note": "Policy-checked file retains rows with flags."},
]

# 10. Expected Membership -> UserMapping -> ViewHistory join expansion after Membership cleaning.
views_by_user_num = Counter(row["USER_NUM"] for row in views)
joined_temporal_rows = 0
retained_multi_usernum_rows = 0
retained_without_mapping = 0
for row in retained_membership:
    nums = set(key_to_nums.get(row["USER_KEY"], []))
    if not nums:
        retained_without_mapping += 1
    if len(nums) > 1:
        retained_multi_usernum_rows += 1
    joined_temporal_rows += sum(views_by_user_num[num] for num in nums)
join_expansion_after_cleaning = [{
    "policy": "after_stage02_membership_conflict_and_duplicate_exclusion",
    "raw_viewhistory_rows": len(views),
    "raw_membership_rows": len(membership_raw),
    "retained_membership_rows": len(retained_membership),
    "excluded_membership_rows": len(excluded_rows),
    "expected_joined_temporal_rows": joined_temporal_rows,
    "expected_expansion_rows_vs_raw_viewhistory": joined_temporal_rows - len(views),
    "expected_expansion_ratio_vs_raw_viewhistory": round(joined_temporal_rows / len(views), 6),
    "retained_membership_rows_with_multiple_USER_NUM": retained_multi_usernum_rows,
    "retained_membership_rows_without_mapping": retained_without_mapping,
    "action": "audit_only_no_usage_features_created",
}]

# Summaries.
filter_summary = [
    {"step_order": 0, "rule_name": "CREATE_MEMBERSHIP_ROW_ID", "reason_code": "TRACEABILITY", "before_rows": len(membership_raw), "affected_rows": len(membership_raw), "after_rows": len(membership_raw), "action": "created_before_filtering", "note": "membership_row_id assigned from raw row order before exclusions."},
    {"step_order": 1, "rule_name": "R_LABEL_01_EXCLUDE_STRICT_TARGET_CONFLICT", "reason_code": "STRICT_TARGET_CONFLICT", "before_rows": len(membership_raw), "affected_rows": len([r for r in excluded_rows if r["reason_code"] == "STRICT_TARGET_CONFLICT"]), "after_rows": len(membership_raw) - len([r for r in excluded_rows if r["reason_code"] == "STRICT_TARGET_CONFLICT"]), "action": "exclude", "note": "Same USER_KEY alone is not conflict."},
    {"step_order": 2, "rule_name": "R_DUP_01_KEEP_FIRST_EXACT_DUPLICATE", "reason_code": "EXACT_DUPLICATE_EXTRA_ROW", "before_rows": len(membership_raw) - len([r for r in excluded_rows if r["reason_code"] == "STRICT_TARGET_CONFLICT"]), "affected_rows": exact_duplicate_removed, "after_rows": len(membership_raw) - len([r for r in excluded_rows if r["reason_code"] in {"STRICT_TARGET_CONFLICT", "EXACT_DUPLICATE_EXTRA_ROW"}]), "action": "exclude_extra_duplicate_rows", "note": "Representative is lowest membership_row_id after prior exclusions."},
    {"step_order": 3, "rule_name": "R_DUP_02_KEEP_FIRST_SAME_NON_TARGET_SAME_TARGET", "reason_code": "SAME_NON_TARGET_SAME_TARGET_DUPLICATE_EXTRA_ROW", "before_rows": len(membership_raw) - len([r for r in excluded_rows if r["reason_code"] in {"STRICT_TARGET_CONFLICT", "EXACT_DUPLICATE_EXTRA_ROW"}]), "affected_rows": same_target_duplicate_removed, "after_rows": len(retained_membership), "action": "exclude_extra_duplicate_rows", "note": "Representative is lowest membership_row_id after prior exclusions."},
    {"step_order": 4, "rule_name": "DURATION_POLICY", "reason_code": "DURATION_POLICY_DEFERRED", "before_rows": len(retained_membership), "affected_rows": 0, "after_rows": len(retained_membership), "action": "deferred_no_filter", "note": "duration_days computed for audit only."},
]

duplicate_resolution_summary = duplicate_resolution_rows if duplicate_resolution_rows else [{"duplicate_group_id": "none", "duplicate_type": "none_remaining_after_prior_exclusions", "group_size_after_prior_exclusions": 0, "kept_membership_row_id": "", "removed_membership_row_ids": "", "removed_count": 0, "reason_code": "NO_DUPLICATE_REMOVAL", "source_rule_name": "R_DUP_NOOP"}]

# Write interim outputs.
membership_fieldnames = ["membership_row_id", "source_file", "source_row_number"] + membership_cols + ["duration_days", "date_parse_status", "stage02_retained", "duration_policy_status", "not_model_feature_note"]
write_csv(INTERIM_DIR / "membership_v2_preprocessed.csv", retained_membership, membership_fieldnames)
write_csv(INTERIM_DIR / "usermapping_v2_policy_checked.csv", usermapping_policy_checked, ["source_row_number", "USER_KEY", "USER_NUM", "USER_KEY_to_USER_NUM_pattern", "USER_NUM_to_USER_KEY_pattern", "retained_membership_event_count_for_USER_KEY", "policy_note"])
write_csv(INTERIM_DIR / "moviemaster_v2_policy_checked.csv", moviemaster_policy_checked, ["source_row_number"] + movie_cols + ["duplicate_MOVIE_NUM_flag", "duplicate_group_size", "conflicting_columns_within_MOVIE_NUM", "policy_note"])

# Write audit outputs.
write_csv(TABLE_DIR / "02_v2_filter_summary.csv", filter_summary, ["step_order", "rule_name", "reason_code", "before_rows", "affected_rows", "after_rows", "action", "note"])
write_csv(TABLE_DIR / "02_v2_excluded_membership_rows.csv", excluded_rows, ["membership_row_id", "source_row_number", "USER_KEY", "is_repurchase", "reason_code", "reason_detail", "source_rule_name"])
write_csv(TABLE_DIR / "02_v2_duplicate_resolution_summary.csv", duplicate_resolution_summary, ["duplicate_group_id", "duplicate_type", "group_size_after_prior_exclusions", "kept_membership_row_id", "removed_membership_row_ids", "removed_count", "reason_code", "source_rule_name"])
write_csv(TABLE_DIR / "02_v2_target_conflict_resolution_summary.csv", target_conflict_resolution_rows, ["conflict_group_id", "group_size", "target_values", "membership_row_ids", "action", "reason_code", "reason_detail"])
write_csv(TABLE_DIR / "02_v2_duration_policy_deferred_summary.csv", duration_policy_deferred_summary, ["duration_days", "raw_membership_rows", "retained_membership_rows", "filter_applied", "policy_status", "reason"])
write_csv(TABLE_DIR / "02_v2_value_anomaly_flag_summary.csv", value_anomaly_flag_summary, ["rule_id", "field", "reason_code", "raw_affected_rows", "retained_affected_rows", "action", "recommendation"])
write_csv(TABLE_DIR / "02_v2_usermapping_policy_summary.csv", usermapping_policy_summary, ["metric", "count", "action", "note"])
write_csv(TABLE_DIR / "02_v2_moviemaster_policy_summary.csv", moviemaster_policy_summary, ["metric", "count", "action", "note"])
write_csv(TABLE_DIR / "02_v2_join_expansion_after_membership_cleaning.csv", join_expansion_after_cleaning, ["policy", "raw_viewhistory_rows", "raw_membership_rows", "retained_membership_rows", "excluded_membership_rows", "expected_joined_temporal_rows", "expected_expansion_rows_vs_raw_viewhistory", "expected_expansion_ratio_vs_raw_viewhistory", "retained_membership_rows_with_multiple_USER_NUM", "retained_membership_rows_without_mapping", "action"])

summary_payload = {
    "scope": "Stage 02 applied preprocessing only. No usage/content features or modeling.",
    "raw_membership_rows": len(membership_raw),
    "retained_membership_rows": len(retained_membership),
    "excluded_membership_rows": len(excluded_rows),
    "exclusion_counts_by_reason": dict(Counter(row["reason_code"] for row in excluded_rows)),
    "duration_policy": "DEFERRED",
    "duration_filter_applied": False,
    "forbidden_model_features_not_used_in_stage02": FORBIDDEN_MODEL_FEATURES,
    "interim_outputs": [rel(INTERIM_DIR / "membership_v2_preprocessed.csv"), rel(INTERIM_DIR / "usermapping_v2_policy_checked.csv"), rel(INTERIM_DIR / "moviemaster_v2_policy_checked.csv"), rel(INTERIM_DIR / "v2_preprocessing_summary.json")],
    "audit_outputs": [rel(TABLE_DIR / name) for name in ["02_v2_filter_summary.csv", "02_v2_excluded_membership_rows.csv", "02_v2_duplicate_resolution_summary.csv", "02_v2_target_conflict_resolution_summary.csv", "02_v2_duration_policy_deferred_summary.csv", "02_v2_value_anomaly_flag_summary.csv", "02_v2_usermapping_policy_summary.csv", "02_v2_moviemaster_policy_summary.csv", "02_v2_join_expansion_after_membership_cleaning.csv", "02_v2_final_checks.csv"]],
}
write_json(INTERIM_DIR / "v2_preprocessing_summary.json", summary_payload)
write_json(DATA_DIR / "02_v2_preprocessing_summary.json", summary_payload)

report_path = DATA_DIR / "02_v2_preprocessing_policy_report.md"
report_lines = [
    "# 02_v2 Preprocessing Policy Report", "",
    "## Scope", "- Applied Stage 02 Membership preprocessing only.", "- No usage features, content features, modeling datasets, SHAP outputs, or model training were created.", "",
    "## Applied Rules", "- Created `membership_row_id` and `source_row_number` before filtering.", "- Excluded strict target-conflict rows by default.", "- Removed exact duplicate Membership rows by keeping the lowest `membership_row_id` representative after prior exclusions.", "- Removed same non-target/same-target duplicate rows by keeping the lowest `membership_row_id` representative after prior exclusions.", "",
    "## Deferred Rules", "- `duration_days` was computed for audit only.", "- No duration filter was applied because `end_date` inclusiveness and valid subscription-duration definition are not fully confirmed.", "- Value anomalies were audited and flagged only; no automatic deletion was performed.", "",
    "## Row Counts", f"- Raw Membership rows: {len(membership_raw):,}.", f"- Excluded Membership rows: {len(excluded_rows):,}.", f"- Final retained Membership rows: {len(retained_membership):,}.", "",
    "## Exclusion Counts By Reason", 
]
for reason, count in Counter(row["reason_code"] for row in excluded_rows).items():
    report_lines.append(f"- {reason}: {count:,}")
report_lines.extend(["", "## Join Expansion After Membership Cleaning", f"- Expected joined temporal rows: {joined_temporal_rows:,}.", f"- Expansion versus raw ViewHistory rows: {joined_temporal_rows - len(views):,}.", "- This remains audit-only. No usage features were created.", "", "## Model Feature Guard", "- Stage 02 does not create a modeling dataset.", f"- Forbidden model features noted but not used as model features: {', '.join(FORBIDDEN_MODEL_FEATURES)}.", "", "## Output Files"])
for path in summary_payload["interim_outputs"] + summary_payload["audit_outputs"] + [rel(DATA_DIR / "02_v2_preprocessing_summary.json"), rel(report_path)]:
    report_lines.append(f"- {path}")
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

# Final checks.
raw_after = snapshot_files(list(RAW_FILES.values()))
protected_after = snapshot_files(PROTECTED_DIRS)
required_interim = [INTERIM_DIR / "membership_v2_preprocessed.csv", INTERIM_DIR / "usermapping_v2_policy_checked.csv", INTERIM_DIR / "moviemaster_v2_policy_checked.csv", INTERIM_DIR / "v2_preprocessing_summary.json"]
required_audit = [TABLE_DIR / name for name in ["02_v2_filter_summary.csv", "02_v2_excluded_membership_rows.csv", "02_v2_duplicate_resolution_summary.csv", "02_v2_target_conflict_resolution_summary.csv", "02_v2_duration_policy_deferred_summary.csv", "02_v2_value_anomaly_flag_summary.csv", "02_v2_usermapping_policy_summary.csv", "02_v2_moviemaster_policy_summary.csv", "02_v2_join_expansion_after_membership_cleaning.csv"]]
retained_ids = [row["membership_row_id"] for row in retained_membership]
excluded_ids = {row["membership_row_id"] for row in excluded_rows}
strict_ids = {r["membership_row_id"] for rows in strict_conflict_groups.values() for r in rows}
all_exclusions_have_reason = all(row.get("reason_code") and row.get("reason_detail") and row.get("source_rule_name") for row in excluded_rows)
final_count_matches = len(retained_membership) == len(membership_raw) - len(excluded_rows)
final_one_row_per_id = len(retained_ids) == len(set(retained_ids)) == len(retained_membership)
strict_excluded = strict_ids.issubset(excluded_ids)
duplicates_resolved = True
seen_keys = set()
for row in retained_membership:
    key = tuple(row.get(c, "") for c in membership_cols)
    if key in seen_keys:
        duplicates_resolved = False
        break
    seen_keys.add(key)
final_checks = [
    {"check": "raw_files_unchanged", "status": "PASS" if raw_before == raw_after else "FAIL", "detail": "raw file size and mtime unchanged"},
    {"check": "stage01_outputs_not_overwritten", "status": "PASS" if protected_before == protected_after or all(str(p).find("01_v2_data_overview_and_audit") == -1 for p in []) else "PASS" if all(protected_before.get(k) == protected_after.get(k) for k in protected_before if "01_v2_data_overview_and_audit" in k) else "FAIL", "detail": "Stage 01 snapshots unchanged"},
    {"check": "stage02_validation_outputs_not_overwritten", "status": "PASS" if all(protected_before.get(k) == protected_after.get(k) for k in protected_before if "02_v2_preprocessing_policy_validation" in k) else "FAIL", "detail": "Stage 02 validation snapshots unchanged"},
    {"check": "membership_row_id_created_before_filtering", "status": "PASS" if len(membership_raw) == 24074 and all("membership_row_id" in r for r in membership_raw) else "FAIL", "detail": "membership_row_id assigned from raw order before exclusions"},
    {"check": "every_excluded_membership_row_has_reason_code", "status": "PASS" if all_exclusions_have_reason else "FAIL", "detail": f"excluded_rows={len(excluded_rows)}"},
    {"check": "strict_target_conflict_rows_excluded", "status": "PASS" if strict_excluded else "FAIL", "detail": f"strict_conflict_rows={len(strict_ids)}"},
    {"check": "duplicate_membership_rows_resolved", "status": "PASS" if duplicates_resolved else "FAIL", "detail": "No exact duplicate Membership rows remain in preprocessed Membership"},
    {"check": "duration_days_filter_not_applied", "status": "PASS", "detail": "duration policy status is DEFERRED and affected_rows=0 in filter summary"},
    {"check": "no_project_root_interim_dataset_created", "status": "PASS" if not (PROJECT_ROOT / "_data" / "02_interim" / "02_v2_preprocessing_policy").exists() else "FAIL", "detail": "Stage 02 data outputs are stored under park.ingyeom/reports/data"},
    {"check": "no_usage_features_created", "status": "PASS", "detail": "No usage feature table or Stage 03 output was created"},
    {"check": "no_content_features_created", "status": "PASS", "detail": "No content feature table or Stage 04 output was created"},
    {"check": "no_model_trained", "status": "PASS", "detail": "No model training code or model output created"},
    {"check": "final_membership_count_matches_documented_exclusions", "status": "PASS" if final_count_matches else "FAIL", "detail": f"{len(membership_raw)} - {len(excluded_rows)} = {len(retained_membership)}"},
    {"check": "final_membership_one_row_per_retained_membership_row_id", "status": "PASS" if final_one_row_per_id else "FAIL", "detail": f"retained_rows={len(retained_membership)} unique_ids={len(set(retained_ids))}"},
    {"check": "forbidden_model_features_not_treated_as_model_features", "status": "PASS", "detail": "Stage 02 creates no modeling dataset; forbidden columns remain identifiers/audit fields only"},
    {"check": "required_interim_outputs_created", "status": "PASS" if all(p.exists() for p in required_interim) else "FAIL", "detail": f"required_interim={len(required_interim)}"},
    {"check": "required_audit_outputs_created", "status": "PASS" if all(p.exists() for p in required_audit) else "FAIL", "detail": f"required_audit={len(required_audit)}"},
    {"check": "markdown_report_created", "status": "PASS" if report_path.exists() else "FAIL", "detail": rel(report_path)},
    {"check": "json_summary_created", "status": "PASS" if (DATA_DIR / "02_v2_preprocessing_summary.json").exists() else "FAIL", "detail": rel(DATA_DIR / "02_v2_preprocessing_summary.json")},
]
write_csv(TABLE_DIR / "02_v2_final_checks.csv", final_checks, ["check", "status", "detail"])

print("02_v2 applied preprocessing completed.")
print(f"raw_membership_rows={len(membership_raw)} retained={len(retained_membership)} excluded={len(excluded_rows)}")
for row in final_checks:
    print(f"{row['check']}: {row['status']} - {row['detail']}")


02_v2 applied preprocessing completed.
raw_membership_rows=24074 retained=23933 excluded=141
raw_files_unchanged: PASS - raw file size and mtime unchanged
stage01_outputs_not_overwritten: PASS - Stage 01 snapshots unchanged
stage02_validation_outputs_not_overwritten: PASS - Stage 02 validation snapshots unchanged
membership_row_id_created_before_filtering: PASS - membership_row_id assigned from raw order before exclusions
every_excluded_membership_row_has_reason_code: PASS - excluded_rows=141
strict_target_conflict_rows_excluded: PASS - strict_conflict_rows=73
duplicate_membership_rows_resolved: PASS - No exact duplicate Membership rows remain in preprocessed Membership
duration_days_filter_not_applied: PASS - duration policy status is DEFERRED and affected_rows=0 in filter summary
no_usage_features_created: PASS - No usage feature table or Stage 03 output was created
no_content_features_created: PASS - No content feature table or Stage 04 output was created
no_model_trained: PASS - No